In [2]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import importlib
import random
import dask.dataframe as dd
import pandas as pd
import numpy as np
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster
import statsmodels.api as sm
from sqlalchemy.orm import aliased
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

# Define trading parameters for OU model
STOP_LOSS_FACTOR = 2.25
DISCOUNT_RATE = 0.01  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
P_VALUE_THRESHOLD = 0.01  # Only trade if p_value < 0.01 (99% confidence)
CLUSTER_TYPE = "local"
N_WORKERS = 4


engine = create_engine(POSTGRES_URL)

In [3]:
max_groups = 5
window_days = 7
window = window_days * 24 * 60
test_days = 90
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

Running pairs trading over period 2025-10-04 to 2026-01-09


In [4]:
include_provider_asset_group_ids = [57]
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).where(
            models.ProviderAssetGroup.is_active.is_(True)
        )
    ).all()
    provider_asset_group_ids = random.sample(provider_asset_group_ids, max_groups)
    provider_asset_group_ids = (
        include_provider_asset_group_ids + provider_asset_group_ids
    )
    provider_asset_group_ids = sorted(provider_asset_group_ids)
    provider_asset_group_ids = list(set(provider_asset_group_ids))
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

Provider asset group ids (count: 6): [937, 2765, 2766, 2290, 57, 3418]


In [ ]:
# Load provider asset group members (filtered)
print("Loading provider asset group members...")
members_data = pd.read_sql(
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    ).where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    ),
    engine,
)
print(f"Members data loaded: {len(members_data)} rows")

# Load market data with pandas (before cluster)
print("Loading market data...")
market_data = pd.read_sql(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    )
    .where(
        models.ProviderAssetMarket.timestamp.between(start_time, end_time),
        models.ProviderAssetMarket.from_asset_id.in_(
            members_data["from_asset_id"].astype(int).unique().tolist()
        ),
        models.ProviderAssetMarket.to_asset_id.in_(
            members_data["to_asset_id"].astype(int).unique().tolist()
        ),
    )
    .order_by(models.ProviderAssetMarket.timestamp),
    engine,
)
print(f"Market data loaded: {len(market_data)} rows")

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="6GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="coiled-cluster",
        n_workers=N_WORKERS,
        region="us-east-1",
        worker_memory="16GB",
        worker_cpu=4,
    )

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
@delayed
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    members_chunk: pd.DataFrame,
    market_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load the pairs trading frame for a chunk of provider asset groups.
    Returns only the essential columns needed for cointegration analysis.

    Args:
        provider_asset_group_ids: List of provider asset group IDs to process
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        members_chunk: Pre-filtered DataFrame of provider asset group members
        market_data: Broadcasted market data DataFrame

    Returns:
        pandas DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Step 1: Generate timeframe using pd.date_range
    time_frame = pd.DataFrame({"timestamp": pd.date_range(start, end, freq="1min")})

    # Step 2: Use passed-in members_chunk (already filtered)
    members = members_chunk

    # Step 3: Cross join
    full_frame = time_frame.merge(members, how="cross")
    full_frame = full_frame.sort_values("timestamp")

    # Step 4: Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame,
        market_data,
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Step 5: Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})
    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    return pairs


def get_pairs_trading_frame(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_ids: list[int],
    members_data: pd.DataFrame,
    market_data_future,
    n_workers: int = 10,
) -> dd.DataFrame:
    """
    Get the pairs trading frame with only essential columns for cointegration analysis.

    Args:
        start: Start datetime (timezone-naive)
        end: End datetime (timezone-naive)
        provider_asset_group_ids: List of provider asset group IDs to process
        members_data: Pre-loaded DataFrame of provider asset group members
        market_data_future: Broadcasted market data future
        n_workers: Number of parallel workers

    Returns:
        Dask DataFrame indexed by provider_asset_group_id with columns:
            - timestamp
            - close_1
            - close_2
    """
    # Split provider asset groups into chunks
    provider_asset_group_ids = sorted(provider_asset_group_ids)
    n_chunks = max(n_workers, len(provider_asset_group_ids))
    group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

    # Create delayed tasks with filtered member chunks
    delayed_dfs = []
    for chunk in group_chunks:
        # Filter members data for this specific chunk
        members_chunk = members_data[
            members_data["provider_asset_group_id"].isin(chunk.tolist())
        ].copy()

        delayed_dfs.append(
            load_pairs_trading_frame_chunk(
                start, end, members_chunk, market_data_future
            )
        )

    # Define minimal schema
    meta = pd.DataFrame(
        {
            "timestamp": pd.Series(dtype="datetime64[ns]"),
            "close_1": pd.Series(dtype="float64"),
            "close_2": pd.Series(dtype="float64"),
        }
    )
    meta.index = pd.Index([], name="provider_asset_group_id", dtype="int64")

    # Convert to Dask DataFrame
    pairs_trading_frame = dd.from_delayed(delayed_dfs, meta=meta)
    pairs_trading_frame = pairs_trading_frame.reset_index()

    return pairs_trading_frame

In [ ]:
pairs_trading_frame = get_pairs_trading_frame(
    start_time,
    end_time,
    provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
def rolling_cointegration(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """Apply rolling cointegration to a DataFrame and return merged result with provider_asset_group_id as index."""
    # Compute rolling cointegration
    cointegration_result = stochastic.RollingCointegration(
        y0=df["close_1"].to_numpy(),
        y1=df["close_2"].to_numpy(),
        window=window,
    ).fit()

    # Create DataFrame with cointegration results indexed by timestamp
    timestamp_values = (
        df["timestamp"].values
        if hasattr(df["timestamp"], "values")
        else df["timestamp"]
    )
    cointegration_df = pd.DataFrame(
        {
            "alpha": cointegration_result.alpha,
            "beta": cointegration_result.beta,
            "pvalue": cointegration_result.pvalue,
            "residual_mean": cointegration_result.residual_mean,
            "residual_std": cointegration_result.residual_std,
        },
        index=timestamp_values,
    )

    # Merge with original DataFrame
    result = df.merge(
        cointegration_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Reset index to drop timestamp index, then set provider_asset_group_id as index
    result = result.reset_index(drop=True)
    result = result.set_index(
        pd.Index([df.name] * len(result), name="provider_asset_group_id")
    )
    return result


pairs_trading_frame = pairs_trading_frame.groupby(
    by="provider_asset_group_id", group_keys=False
)[["timestamp", "close_1", "close_2"]].apply(
    lambda df: rolling_cointegration(df, window),
    meta=pd.DataFrame(
        data={
            "timestamp": pd.Series([], dtype="datetime64[ns]"),
            "close_1": pd.Series([], dtype=float),
            "close_2": pd.Series([], dtype=float),
            "alpha": pd.Series([], dtype=float),
            "beta": pd.Series([], dtype=float),
            "pvalue": pd.Series([], dtype=float),
            "residual_mean": pd.Series([], dtype=float),
            "residual_std": pd.Series([], dtype=float),
        },
        index=pd.Index([], name="provider_asset_group_id", dtype="int64"),
    ),
)

In [ ]:
pairs_trading_frame["loss_level"] = (
    pairs_trading_frame["residual_mean"]
    - pairs_trading_frame["residual_std"] * STOP_LOSS_FACTOR
)

In [ ]:
def rolling_ornstein_uhlenbeck(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """Apply rolling Ornstein-Uhlenbeck to a DataFrame and return merged result with provider_asset_group_id as index."""
    # Compute rolling Ornstein-Uhlenbeck
    ou_result = stochastic.RollingOrnsteinUhlenbeck(
        alpha=df["alpha"].to_numpy(),
        beta=df["beta"].to_numpy(),
        pvalue=df["pvalue"].to_numpy(),
        y0=df["close_1"].to_numpy(),
        y1=df["close_2"].to_numpy(),
        window=window,
        pvalue_threshold=P_VALUE_THRESHOLD,
    ).fit()

    # Create DataFrame with OU results indexed by timestamp
    timestamp_values = (
        df["timestamp"].values
        if hasattr(df["timestamp"], "values")
        else df["timestamp"]
    )
    ou_df = pd.DataFrame(
        {
            "mu": ou_result.mu,
            "sigma": ou_result.sigma,
            "theta": ou_result.theta,
            "half_life": ou_result.half_life,
        },
        index=timestamp_values,
    )

    # Merge with original DataFrame
    result = df.merge(
        ou_df,
        left_on="timestamp",
        right_index=True,
        how="inner",
    )

    # Reset index to drop timestamp index, then set provider_asset_group_id as index
    result = result.reset_index(drop=True)
    result = result.set_index(
        pd.Index([df.name] * len(result), name="provider_asset_group_id")
    )
    return result


# After first groupby, provider_asset_group_id is in index
pairs_trading_frame = pairs_trading_frame.groupby(
    by="provider_asset_group_id", group_keys=False
)[
    [
        "timestamp",
        "close_1",
        "close_2",
        "alpha",
        "beta",
        "pvalue",
        "residual_mean",
        "residual_std",
        "loss_level",
    ]
].apply(
    lambda df: rolling_ornstein_uhlenbeck(df, window),
    meta=pd.DataFrame(
        {
            "timestamp": pd.Series([], dtype="datetime64[ns]"),
            "close_1": pd.Series([], dtype=float),
            "close_2": pd.Series([], dtype=float),
            "alpha": pd.Series([], dtype=float),
            "beta": pd.Series([], dtype=float),
            "pvalue": pd.Series([], dtype=float),
            "residual_mean": pd.Series([], dtype=float),
            "residual_std": pd.Series([], dtype=float),
            "loss_level": pd.Series([], dtype=float),
            "mu": pd.Series([], dtype=float),
            "sigma": pd.Series([], dtype=float),
            "theta": pd.Series([], dtype=float),
            "half_life": pd.Series([], dtype=float),
        },
        index=pd.Index([], name="provider_asset_group_id", dtype="int64"),
    ),
)

In [ ]:
pairs_trading_frame.persist()

In [ ]:
pairs_trading_frame_computed = pairs_trading_frame.compute()
pairs_trading_frame_computed

In [ ]:
pairs_trading_frame_computed.to_parquet("pairs_trading_frame_computed.parquet")

In [ ]:
pairs_trading_frame_computed = pd.read_parquet("pairs_trading_frame_computed.parquet")
pairs_trading_frame_computed

In [ ]:
sample = pairs_trading_frame_computed.reset_index(level="provider_asset_group_id")
sample = sample.loc[
    (sample["pvalue"] < P_VALUE_THRESHOLD) & (sample["provider_asset_group_id"] == 57)
]
sample

In [ ]:
def get_exit_level(
    mu: np.ndarray,
    sigma: np.ndarray,
    theta: np.ndarray,
    r: np.ndarray,
    c: np.ndarray,
    max_iter: int = 50,
    tol: float = 1e-7,
    max_initial_shift: int = 100,
) -> np.ndarray:
    """
    Find the root for b such that f(b, mu, sigma, theta, r, c) = 0 using Newton's method (vectorized).
    Ensures no infinite loops by capping the number of initial guess shifts.

    Args:
        mu, sigma, theta, r, c (np.ndarray): parameter arrays (same length)
        max_iter (int): max number of Newton iterations
        tol (float): convergence tolerance
        max_initial_shift (int): max number of initial guess increments

    Returns:
        np.ndarray: solution for b for each row

    Raises:
        RuntimeError: If suitable initial guess could not be found (after max_initial_shift increments)
    """

    # Define the function f(b)
    def f_exit_level(
        b: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
    ):
        return (b - c) * stochastic.OrnsteinUhlenbeck.F_prime(
            b, mu, sigma, theta, r, use_analytical=True
        ) - stochastic.OrnsteinUhlenbeck.F(b, mu, sigma, theta, r, use_analytical=True)

    # Define the derivative of f(b)
    def f_prime_exit_level(
        b: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        h: float = 1e-6,
    ):
        return (
            f_exit_level(b + h, mu, sigma, theta, r, c)
            - f_exit_level(b - h, mu, sigma, theta, r, c)
        ) / (2 * h)

    initial_guess = np.copy(theta)
    mask = f_exit_level(initial_guess, mu, sigma, theta, r, c) < 0

    shift_counter = 0
    # Cap the number of shifts to ensure no infinite loop
    while np.any(mask) and shift_counter < max_initial_shift:
        initial_guess[mask] += 2 * sigma[mask]
        shift_counter += 1
        mask = f_exit_level(initial_guess, mu, sigma, theta, r, c) < 0
    if np.any(mask):
        raise RuntimeError(
            f"Failed to find suitable initial guess for all points after {max_initial_shift} increments."
        )

    # Newton's method
    b = initial_guess
    converged = np.zeros_like(b, dtype=bool)
    for i in range(max_iter):
        mask = ~converged
        if not np.any(mask):
            break
        # Only compute f and f_prime for the non-converged values
        f_val = np.zeros_like(b)
        f_prime_val = np.ones_like(
            b
        )  # Avoid division by zero if not used; will be overwritten for masked vals
        f_val[mask] = f_exit_level(
            b[mask], mu[mask], sigma[mask], theta[mask], r[mask], c[mask]
        )
        f_prime_val[mask] = f_prime_exit_level(
            b[mask], mu[mask], sigma[mask], theta[mask], r[mask], c[mask]
        )
        # Prevent division by zero and infinite updates
        update_mask = (f_prime_val[mask] != 0) & np.isfinite(f_prime_val[mask])
        # Only update values where f' is valid
        indices_to_update = np.flatnonzero(mask)[update_mask]
        b[indices_to_update] = (
            b[indices_to_update]
            - f_val[indices_to_update] / f_prime_val[indices_to_update]
        )
        # Update convergence status only for these points
        converged[mask] = (np.abs(f_val[mask]) < tol) | ~update_mask
    # Warn if some have not converged
    if not np.all(converged):
        import warnings

        warnings.warn(
            f"{np.sum(~converged)} roots did not converge within {max_iter} iterations."
        )
    return b


# Example usage:
sample["exit_level"] = get_exit_level(
    sample["mu"].to_numpy(),
    sample["sigma"].to_numpy(),
    sample["theta"].to_numpy(),
    np.ones(len(sample)) * DISCOUNT_RATE,
    np.ones(len(sample)) * TRANSACTION_COST,
    max_iter=100,
)
sample

In [ ]:
def get_entry_level(
    mu: np.ndarray,
    sigma: np.ndarray,
    theta: np.ndarray,
    r: np.ndarray,
    c: np.ndarray,
    exit_level: np.ndarray,
    max_iter: int = 50,
    tol: float = 1e-7,
    max_initial_shift: int = 100,
) -> np.ndarray:
    """
    Find the root for b such that f(b, mu, sigma, theta, r, c) = 0 using Newton's method (vectorized).
    Ensures no infinite loops by capping the number of initial guess shifts.

    Args:
        mu, sigma, theta, r, c (np.ndarray): parameter arrays (same length)
        max_iter (int): max number of Newton iterations
        tol (float): convergence tolerance
        max_initial_shift (int): max number of initial guess increments

    Returns:
        np.ndarray: solution for b for each row

    Raises:
        RuntimeError: If suitable initial guess could not be found (after max_initial_shift increments)
    """

    # Define the function f(b)
    def f_entry_level(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        exit_level: float | np.ndarray,
    ):
        x_neg = -x  # We
        return (
            stochastic.OrnsteinUhlenbeck.G(
                x_neg, mu, sigma, theta, r, use_analytical=True
            )
            * (
                stochastic.OrnsteinUhlenbeck.V_prime(
                    x_neg, mu, sigma, theta, r, c, exit_level, use_analytical=True
                )
                - 1
            )
        ) - (
            stochastic.OrnsteinUhlenbeck.G_prime(
                x_neg, mu, sigma, theta, r, use_analytical=True
            )
            * (
                stochastic.OrnsteinUhlenbeck.V(
                    x_neg, mu, sigma, theta, r, c, exit_level, use_analytical=True
                )
                - x_neg
                - c
            )
        )

    def f_prime_entry_level(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        exit_level: float | np.ndarray,
        h: float = 1e-6,
    ):
        return (
            f_entry_level(x + h, mu, sigma, theta, r, c, exit_level)
            - f_entry_level(x - h, mu, sigma, theta, r, c, exit_level)
        ) / (2 * h)

    initial_guess = np.copy(theta)
    mask = f_entry_level(initial_guess, mu, sigma, theta, r, c, exit_level) < 0

    shift_counter = 0
    # Cap the number of shifts to ensure no infinite loop
    while np.any(mask) and shift_counter < max_initial_shift:
        initial_guess[mask] += 2 * sigma[mask]
        shift_counter += 1
        mask = f_entry_level(initial_guess, mu, sigma, theta, r, c, exit_level) < 0
    if np.any(mask):
        raise RuntimeError(
            f"Failed to find suitable initial guess for all points after {max_initial_shift} increments."
        )

    # Newton's method
    d = initial_guess
    converged = np.zeros_like(d, dtype=bool)
    for i in range(max_iter):
        mask = ~converged
        if not np.any(mask):
            break
        # Only compute f and f_prime for the non-converged values
        f_val = np.zeros_like(d)
        f_prime_val = np.ones_like(
            d
        )  # Avoid division by zero if not used; will be overwritten for masked vals
        f_val[mask] = f_entry_level(
            d[mask],
            mu[mask],
            sigma[mask],
            theta[mask],
            r[mask],
            c[mask],
            exit_level[mask],
        )
        f_prime_val[mask] = f_prime_entry_level(
            d[mask],
            mu[mask],
            sigma[mask],
            theta[mask],
            r[mask],
            c[mask],
            exit_level[mask],
        )
        # Prevent division by zero and infinite updates
        update_mask = (f_prime_val[mask] != 0) & np.isfinite(f_prime_val[mask])
        # Only update values where f' is valid
        indices_to_update = np.flatnonzero(mask)[update_mask]
        d[indices_to_update] = (
            d[indices_to_update]
            - f_val[indices_to_update] / f_prime_val[indices_to_update]
        )
        # Update convergence status only for these points
        converged[mask] = (np.abs(f_val[mask]) < tol) | ~update_mask
    # Warn if some have not converged
    if not np.all(converged):
        import warnings

        warnings.warn(
            f"{np.sum(~converged)} roots did not converge within {max_iter} iterations."
        )
    return -d


# Example usage:
sample["entry_level"] = get_entry_level(
    sample["mu"].to_numpy(),
    sample["sigma"].to_numpy(),
    sample["theta"].to_numpy(),
    np.ones(len(sample)) * DISCOUNT_RATE,
    np.ones(len(sample)) * TRANSACTION_COST,
    sample["exit_level"].to_numpy(),
)
sample

In [ ]:
STOP_LOSS_PERCENTAGE = 3.5
sample["loss_level"] = sample["theta"] + (sample["entry_level"] - sample["theta"]) * (
    1 + STOP_LOSS_PERCENTAGE
)
sample

In [ ]:
sample["entry_level_1"] = sample["entry_level"]
sample["exit_level_1"] = sample["exit_level"]
sample["loss_level_1"] = sample["loss_level"]
sample["entry_level_2"] = 2 * sample["theta"] - sample["entry_level_1"]
sample["exit_level_2"] = 2 * sample["theta"] - sample["exit_level_1"]
sample["loss_level_2"] = 2 * sample["theta"] - sample["loss_level_1"]
sample

In [ ]:
def get_exit_level_with_loss_level(
    mu: np.ndarray,
    sigma: np.ndarray,
    theta: np.ndarray,
    r: np.ndarray,
    c: np.ndarray,
    loss_level: np.ndarray,
    max_iter: int = 50,
    tol: float = 1e-7,
    max_initial_shift: int = 100,
) -> np.ndarray:
    """
    Find the root for b such that f(b, mu, sigma, theta, r, c) = 0 using Newton's method (vectorized).
    Ensures no infinite loops by capping the number of initial guess shifts.

    Args:
        mu, sigma, theta, r, c (np.ndarray): parameter arrays (same length)
        max_iter (int): max number of Newton iterations
        tol (float): convergence tolerance
        max_initial_shift (int): max number of initial guess increments

    Returns:
        np.ndarray: solution for b for each row

    Raises:
        RuntimeError: If suitable initial guess could not be found (after max_initial_shift increments)
    """

    # Define the function f(b)
    def f_exit_level_with_loss_level(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        loss_level: float | np.ndarray,
        h: float = 1e-6,
    ) -> float | np.ndarray:
        neg_x = -x
        F_L = stochastic.OrnsteinUhlenbeck.F(
            loss_level, mu, sigma, theta, r, use_analytical=True
        )
        G_L = stochastic.OrnsteinUhlenbeck.G(
            loss_level, mu, sigma, theta, r, use_analytical=True
        )
        G_b = stochastic.OrnsteinUhlenbeck.G(
            neg_x, mu, sigma, theta, r, use_analytical=True
        )
        F_prime_b = stochastic.OrnsteinUhlenbeck.F_prime(
            neg_x,
            mu,
            sigma,
            theta,
            r,
            h=h,
            use_analytical=True,
        )
        F_b = stochastic.OrnsteinUhlenbeck.F(
            neg_x, mu, sigma, theta, r, use_analytical=True
        )
        G_prime_b = stochastic.OrnsteinUhlenbeck.G_prime(
            neg_x,
            mu,
            sigma,
            theta,
            r,
            h=h,
            use_analytical=True,
        )

        f_left = ((loss_level - c) * G_b - (neg_x - c) * F_prime_b) + (
            (neg_x - c) * F_L - (loss_level - c) * F_b
        ) * G_prime_b

        f_right = G_b * F_L - G_L * F_b

        return f_left - f_right

    # Define the derivative of f(b)
    def f_prime_exit_level_with_loss_level(
        x: float | np.ndarray,
        mu: float | np.ndarray,
        sigma: float | np.ndarray,
        theta: float | np.ndarray,
        r: float | np.ndarray,
        c: float | np.ndarray,
        loss_level: float | np.ndarray,
        h: float = 1e-6,
    ):
        return (
            f_exit_level_with_loss_level(x + h, mu, sigma, theta, r, c, loss_level)
            - f_exit_level_with_loss_level(x - h, mu, sigma, theta, r, c, loss_level)
        ) / (2 * h)

    initial_guess = np.copy(theta)
    mask = (
        f_exit_level_with_loss_level(initial_guess, mu, sigma, theta, r, c, loss_level)
        < 0
    )

    shift_counter = 0
    # Cap the number of shifts to ensure no infinite loop
    while np.any(mask) and shift_counter < max_initial_shift:
        initial_guess[mask] += 2 * sigma[mask]
        shift_counter += 1
        mask = (
            f_exit_level_with_loss_level(
                initial_guess, mu, sigma, theta, r, c, loss_level
            )
            < 0
        )
    if np.any(mask):
        raise RuntimeError(
            f"Failed to find suitable initial guess for all points after {max_initial_shift} increments."
        )

    # Newton's method
    b = initial_guess
    converged = np.zeros_like(b, dtype=bool)
    for i in range(max_iter):
        mask = ~converged
        if not np.any(mask):
            break
        # Only compute f and f_prime for the non-converged values
        f_val = np.zeros_like(b)
        f_prime_val = np.ones_like(
            b
        )  # Avoid division by zero if not used; will be overwritten for masked vals
        f_val[mask] = f_exit_level_with_loss_level(
            b[mask],
            mu[mask],
            sigma[mask],
            theta[mask],
            r[mask],
            c[mask],
            loss_level[mask],
        )
        f_prime_val[mask] = f_prime_exit_level_with_loss_level(
            b[mask],
            mu[mask],
            sigma[mask],
            theta[mask],
            r[mask],
            c[mask],
            loss_level[mask],
        )
        # Prevent division by zero and infinite updates
        update_mask = (f_prime_val[mask] != 0) & np.isfinite(f_prime_val[mask])
        # Only update values where f' is valid
        indices_to_update = np.flatnonzero(mask)[update_mask]
        b[indices_to_update] = (
            b[indices_to_update]
            - f_val[indices_to_update] / f_prime_val[indices_to_update]
        )
        # Update convergence status only for these points
        converged[mask] = (np.abs(f_val[mask]) < tol) | ~update_mask
    # Warn if some have not converged
    if not np.all(converged):
        import warnings

        warnings.warn(
            f"{np.sum(~converged)} roots did not converge within {max_iter} iterations."
        )
    return b


# Example usage:
sample["exit_level_with_loss_level"] = get_exit_level_with_loss_level(
    sample["mu"].to_numpy(),
    sample["sigma"].to_numpy(),
    sample["theta"].to_numpy(),
    np.ones(len(sample)) * DISCOUNT_RATE,
    np.ones(len(sample)) * TRANSACTION_COST,
    sample["loss_level"].to_numpy(),
)
sample

In [ ]:
def f_prime_exit_level_with_loss_level(
    x: float | np.ndarray,
    mu: float | np.ndarray,
    sigma: float | np.ndarray,
    theta: float | np.ndarray,
    r: float | np.ndarray,
    c: float | np.ndarray,
    loss_level: float | np.ndarray,
    h: float = 1e-6,
):
    return (
        f_exit_level_with_loss_level(x + h, mu, sigma, theta, r, c, loss_level)
        - f_exit_level_with_loss_level(x - h, mu, sigma, theta, r, c, loss_level)
    ) / (2 * h)

In [ ]:
sample

In [ ]:
import matplotlib.pyplot as plt

test = sample.iloc[0]
mu = test["mu"]
sigma = test["sigma"]
theta = test["theta"]
r = DISCOUNT_RATE
c = TRANSACTION_COST
loss_level = test["loss_level"]
x = -1 * np.linspace(-0.14, -0.05, 1000)
y = f_exit_level_with_loss_level(x, mu, sigma, theta, r, c, loss_level)
plt.axhline(0, color="black", linewidth=0.5)
plt.plot(x, y)
plt.show()

In [ ]:
df_backtest = pairs_trading_frame_computed.reset_index(level="provider_asset_group_id")
df_backtest = df_backtest.loc[(df_backtest["provider_asset_group_id"] == 57)]
df_backtest = df_backtest.merge(
    sample[
        [
            "provider_asset_group_id",
            "timestamp",
            "entry_level_1",
            "exit_level_1",
            "loss_level_1",
            "entry_level_2",
            "exit_level_2",
            "loss_level_2",
        ]
    ],
    on=["provider_asset_group_id", "timestamp"],
    how="left",
)
df_backtest = df_backtest.loc[df_backtest["alpha"].notna()]
df_backtest["spread"] = (
    df_backtest["close_1"]
    - df_backtest["alpha"]
    - df_backtest["beta"] * df_backtest["close_2"]
)
df_backtest

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=1)

fig.update_layout(
    title="Spread and Entry/Exit Levels", legend=dict(font=dict(family="serif"))
)

fig.add_trace(
    go.Scatter(
        x=df_backtest["timestamp"], y=df_backtest["spread"], mode="lines", name="Spread"
    )
)
fig.add_trace(
    go.Scatter(
        x=df_backtest["timestamp"],
        y=df_backtest["theta"],
        mode="lines",
        name="Theta (θ)",
    )
)
fig.add_trace(
    go.Scatter(
        x=df_backtest["timestamp"],
        y=df_backtest["entry_level_1"],
        mode="lines",
        name="Entry Level (b)",
    )
)
fig.add_trace(
    go.Scatter(
        x=df_backtest["timestamp"],
        y=df_backtest["exit_level_1"],
        mode="lines",
        name="Exit Level (d)",
    )
)
fig.add_trace(
    go.Scatter(
        x=df_backtest["timestamp"],
        y=df_backtest["loss_level_1"],
        mode="lines",
        name="Loss Level (l)",
    )
)

fig.show()

In [ ]:
# Backtest the pairs trading strategy using df_backtest (pandas version)
# Strategy: Greedily enter whichever direction signals first, one trade at a time

print("Configuration: Greedy Bidirectional Trading - Pandas DataFrame version")
print("  - Will enter whichever direction signals first")
print("  - Only one position at a time")
print("  - Direction 1 (spread down): short close_2, long close_1")
print("  - Direction 2 (spread up): short close_1, long close_2")

import pandas as pd
import numpy as np

# Check that df_backtest has required columns
required_cols = [
    "timestamp",
    "spread",
    "pvalue",
    "entry_level_1",
    "exit_level_1",
    "loss_level_1",
    "entry_level_2",
    "exit_level_2",
    "loss_level_2",
    "close_1",
    "close_2",
]
missing_cols = [col for col in required_cols if col not in df_backtest.columns]
if missing_cols:
    print(f"ERROR: df_backtest is missing columns: {missing_cols}")
    print(f"Available columns: {df_backtest.columns}")
else:
    print(f"df_backtest has {len(df_backtest)} rows")

    initial_cash = 100000
    cash = initial_cash
    in_position = False
    current_direction = None
    shares_short = 0
    shares_long = 0
    entry_price_short = 0
    entry_price_long = 0

    last_spread = None
    trade_history = []
    skipped_rows = 0

    # Helper function to check if values are valid (pandas/NumPy)
    def is_valid(val):
        return val is not None and not pd.isna(val)

    for i in range(len(df_backtest)):
        # .iloc returns a Series, .at is for fast scalar access
        row = df_backtest.iloc[i]

        p_value = row["pvalue"]
        spread = row["spread"]
        entry_level_1 = row["entry_level_1"]
        entry_level_2 = row["entry_level_2"]
        exit_level_1 = row["exit_level_1"]
        exit_level_2 = row["exit_level_2"]
        loss_level_1 = row["loss_level_1"]
        loss_level_2 = row["loss_level_2"]
        price_1 = row["close_1"]
        price_2 = row["close_2"]
        timestamp = row["timestamp"]

        # Check validity
        p_valid = is_valid(p_value) and p_value <= P_VALUE_THRESHOLD
        dir1_levels_valid = all(
            is_valid(v) for v in [spread, entry_level_1, exit_level_1, loss_level_1]
        )
        dir2_levels_valid = all(
            is_valid(v) for v in [spread, entry_level_2, exit_level_2, loss_level_2]
        )

        if not in_position:
            entered = False

            # Direction 1: spread crosses DOWN below entry_level_1
            if not entered and p_valid and dir1_levels_valid:
                loss_level_1_adj = loss_level_1
                if entry_level_1 <= loss_level_1:
                    loss_level_1_adj = entry_level_1 - 0.0001
                if (
                    last_spread is not None
                    and last_spread >= entry_level_1
                    and spread < entry_level_1
                ):
                    price_short, price_long = price_2, price_1
                    position_value = initial_cash * 0.5
                    shares_short = position_value / price_short
                    shares_long = position_value / price_long
                    cash -= (shares_short * price_short * TRANSACTION_COST) + (
                        shares_long * price_long * TRANSACTION_COST
                    )
                    entry_price_short = price_short
                    entry_price_long = price_long
                    in_position = True
                    current_direction = 1
                    entered = True

                    trade_history.append(
                        {
                            "type": "entry",
                            "direction": 1,
                            "index": i,
                            "timestamp": timestamp,
                            "spread": spread,
                            "price_short": price_short,
                            "price_long": price_long,
                            "entry_level": entry_level_1,
                            "exit_level": exit_level_1,
                            "loss_level": loss_level_1,
                            "shares_short": shares_short,
                            "shares_long": shares_long,
                            "pvalue": p_value,
                        }
                    )
            # Direction 2: spread crosses UP above entry_level_2
            if not entered and p_valid and dir2_levels_valid:
                loss_level_2_adj = loss_level_2
                if entry_level_2 >= loss_level_2:
                    loss_level_2_adj = entry_level_2 + 0.0001
                if (
                    last_spread is not None
                    and last_spread <= entry_level_2
                    and spread > entry_level_2
                ):
                    price_short, price_long = price_1, price_2
                    position_value = initial_cash * 0.5
                    shares_short = position_value / price_short
                    shares_long = position_value / price_long
                    cash -= (shares_short * price_short * TRANSACTION_COST) + (
                        shares_long * price_long * TRANSACTION_COST
                    )
                    entry_price_short = price_short
                    entry_price_long = price_long
                    in_position = True
                    current_direction = 2
                    entered = True

                    trade_history.append(
                        {
                            "type": "entry",
                            "direction": 2,
                            "index": i,
                            "timestamp": timestamp,
                            "spread": spread,
                            "price_short": price_short,
                            "price_long": price_long,
                            "entry_level": entry_level_2,
                            "exit_level": exit_level_2,
                            "loss_level": loss_level_2,
                            "shares_short": shares_short,
                            "shares_long": shares_long,
                            "pvalue": p_value,
                        }
                    )
            if not entered and not p_valid:
                skipped_rows += 1

        else:
            if current_direction == 1:
                exit_level = exit_level_1
                loss_level = loss_level_1
                price_short, price_long = price_2, price_1
                levels_valid = dir1_levels_valid
            else:
                exit_level = exit_level_2
                loss_level = loss_level_2
                price_short, price_long = price_1, price_2
                levels_valid = dir2_levels_valid

            if not is_valid(spread) or not levels_valid:
                if is_valid(spread):
                    last_spread = spread
                continue

            exit_signal = False
            exit_reason = None

            if current_direction == 1:
                if (
                    last_spread is not None
                    and last_spread <= exit_level
                    and spread > exit_level
                ):
                    exit_signal = True
                    exit_reason = "take_profit"
                elif (
                    last_spread is not None
                    and last_spread >= loss_level
                    and spread < loss_level
                ):
                    exit_signal = True
                    exit_reason = "stop_loss"
            else:
                if (
                    last_spread is not None
                    and last_spread >= exit_level
                    and spread < exit_level
                ):
                    exit_signal = True
                    exit_reason = "take_profit"
                elif (
                    last_spread is not None
                    and last_spread <= loss_level
                    and spread > loss_level
                ):
                    exit_signal = True
                    exit_reason = "stop_loss"

            if exit_signal:
                pnl_short = (entry_price_short - price_short) * shares_short
                pnl_long = (price_long - entry_price_long) * shares_long
                total_pnl = pnl_short + pnl_long
                cash += total_pnl
                cash -= (shares_short * price_short * TRANSACTION_COST) + (
                    shares_long * price_long * TRANSACTION_COST
                )
                trade_history.append(
                    {
                        "type": "exit",
                        "direction": current_direction,
                        "index": i,
                        "timestamp": timestamp,
                        "spread": spread,
                        "price_short": price_short,
                        "price_long": price_long,
                        "exit_level": exit_level,
                        "loss_level": loss_level,
                        "exit_reason": exit_reason,
                        "pnl_short": pnl_short,
                        "pnl_long": pnl_long,
                        "total_pnl": total_pnl,
                        "pvalue": p_value,
                    }
                )
                in_position = False
                current_direction = None
                shares_short = 0
                shares_long = 0
                entry_price_short = 0
                entry_price_long = 0

        if is_valid(spread):
            last_spread = spread

    # Calculate final portfolio value
    final_portfolio_value = cash
    if in_position:
        if current_direction == 1:
            price_short, price_long = price_2, price_1
        else:
            price_short, price_long = price_1, price_2
        pnl_short = (entry_price_short - price_short) * shares_short
        pnl_long = (price_long - entry_price_long) * shares_long
        final_portfolio_value += pnl_short + pnl_long
        print(
            f"\nStill in position (Direction {current_direction}) at end. Unrealized P&L: {pnl_short + pnl_long:.2f}"
        )

    # Calculate returns
    total_return = final_portfolio_value - initial_cash
    return_pct = (total_return / initial_cash) * 100

    # Count trades by direction
    entries = [t for t in trade_history if t["type"] == "entry"]
    entries_dir1 = len([t for t in entries if t["direction"] == 1])
    entries_dir2 = len([t for t in entries if t["direction"] == 2])

    exits = [t for t in trade_history if t["type"] == "exit"]
    take_profits = len([t for t in exits if t["exit_reason"] == "take_profit"])
    stop_losses = len([t for t in exits if t["exit_reason"] == "stop_loss"])

    print("\n===== Backtest Results (Greedy Bidirectional) =====")
    print("\nPerformance:")
    print(f"  Initial Cash: ${initial_cash:,.2f}")
    print(f"  Final Portfolio Value: ${final_portfolio_value:,.2f}")
    print(f"  Total Return: ${total_return:,.2f} ({return_pct:.2f}%)")
    print("\nTrading Activity:")
    print(f"  Total Trades: {len(entries)}")
    print(
        f"    - Direction 1 (spread down, short close_2, long close_1): {entries_dir1}"
    )
    print(f"    - Direction 2 (spread up, short close_1, long close_2): {entries_dir2}")
    print(f"  Exits: {len(exits)}")
    print(f"    - Take Profits: {take_profits}")
    print(f"    - Stop Losses: {stop_losses}")
    print(f"  Rows Skipped (no valid p_value): {skipped_rows}")
    print(
        "\nNote: Exits can occur even when p_value is invalid (using forward-filled spread)"
    )

    # Convert trade_history to Pandas dataframe
    if trade_history:
        trades_df = pd.DataFrame(trade_history)
    else:
        trades_df = pd.DataFrame()

In [ ]:
# Resample data to reduce points for plotting
# Target: max 10000 points or resample to 5-minute intervals, whichever gives fewer points
MAX_PLOT_POINTS = 10000
RESAMPLE_INTERVAL = "5min"  # 5 minutes in pandas freq string

original_length = len(df_backtest)
print(f"Original data points: {original_length}")

if original_length > MAX_PLOT_POINTS:
    df_backtest_plot = df_backtest.copy()
    # Ensure timestamp is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_backtest_plot["timestamp"]):
        df_backtest_plot["timestamp"] = pd.to_datetime(df_backtest_plot["timestamp"])
    df_backtest_plot = df_backtest_plot.sort_values("timestamp")

    # Try resampling by time interval first using pandas
    try:
        df_backtest_plot = (
            df_backtest_plot.set_index("timestamp")
            .resample(RESAMPLE_INTERVAL)
            .agg(
                {
                    "spread": "mean",
                    "pvalue": "first",
                    "theta": "first",
                    "entry_level_1": "first",
                    "exit_level_1": "first",
                    "loss_level_1": "first",
                    "entry_level_2": "first",
                    "exit_level_2": "first",
                    "loss_level_2": "first",
                }
            )
        )
        # Use timestamp as a column after resampling
        df_backtest_plot = df_backtest_plot.reset_index()

        # If still too many points, downsample further
        if len(df_backtest_plot) > MAX_PLOT_POINTS:
            step = max(1, len(df_backtest_plot) // MAX_PLOT_POINTS)
            df_backtest_plot = df_backtest_plot.iloc[::step].reset_index(drop=True)

    except Exception as e:
        print(f"Time-based resampling failed: {e}, using simple downsampling")
        # Fallback: simple downsampling by taking every Nth row
        step = max(1, original_length // MAX_PLOT_POINTS)
        df_backtest_plot = df_backtest_plot.iloc[::step].reset_index(drop=True)

    print(
        f"Resampled data points: {len(df_backtest_plot)} ({100 * len(df_backtest_plot) / original_length:.1f}% of original)"
    )
    df_backtest = df_backtest_plot
else:
    print("Data points within limit, no resampling needed")

In [ ]:
# Use the resampled df_backtest_plot if available, otherwise fall back to df_backtest
if "df_backtest_plot" in locals():
    plot_df = df_backtest_plot
else:
    plot_df = df_backtest

# Check that we have the required data
if plot_df is None or len(plot_df) == 0:
    print("ERROR: No data available for plotting. Please run the previous cells first.")
elif "trades_df" not in locals() or len(trades_df) == 0:
    print("WARNING: No trades found. Plotting levels only.")
    trades_df = pd.DataFrame()

# Color scheme - different colors for each direction
COLORS = {
    # Shared
    "spread": "rgba(50, 50, 50, 0.9)",  # Dark gray for the shared spread
    "mean": "purple",
    # Direction 1 (blues/greens)
    "dir1_entry": "rgba(0, 128, 0, 0.8)",  # Green
    "dir1_exit": "rgba(0, 100, 200, 0.8)",  # Blue
    "dir1_loss": "rgba(0, 180, 180, 0.8)",  # Teal
    "dir1_trade_entry": "green",
    "dir1_trade_tp": "blue",
    "dir1_trade_sl": "teal",
    "dir1_connector": "rgba(0, 128, 0, 0.4)",
    # Direction 2 (reds/oranges)
    "dir2_entry": "rgba(200, 50, 50, 0.8)",  # Red
    "dir2_exit": "rgba(255, 140, 0, 0.8)",  # Orange
    "dir2_loss": "rgba(180, 0, 180, 0.8)",  # Magenta
    "dir2_trade_entry": "red",
    "dir2_trade_tp": "orange",
    "dir2_trade_sl": "magenta",
    "dir2_connector": "rgba(200, 50, 50, 0.4)",
}

# Use the possibly resampled DataFrame
timestamps = plot_df["timestamp"].to_list()

# Single pvalue and spread (used for both directions)
p_values = plot_df["pvalue"].to_list()
spread = plot_df["spread"].to_list()
# Allow ou_theta or theta for compatibility
theta_col = (
    "ou_theta"
    if "ou_theta" in plot_df.columns
    else ("theta" if "theta" in plot_df.columns else None)
)
theta_values = plot_df[theta_col].to_list() if theta_col else None

# Direction 1 levels
entry_levels_1 = plot_df["entry_level_1"].to_list()
exit_levels_1 = plot_df["exit_level_1"].to_list()
loss_levels_1 = plot_df["loss_level_1"].to_list()

# Direction 2 levels
entry_levels_2 = plot_df["entry_level_2"].to_list()
exit_levels_2 = plot_df["exit_level_2"].to_list()
loss_levels_2 = plot_df["loss_level_2"].to_list()


# Helper function to mask invalid values based on single pvalue
def mask_invalid_values(p_values, data_list):
    """Mask values where pvalue is invalid."""
    result = list(data_list)
    for i, pv in enumerate(p_values):
        is_invalid = pv is None or (
            isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD)
        )
        if is_invalid:
            result[i] = None
    return result


# Mask invalid values (single pvalue applies to all)
spread_masked = mask_invalid_values(p_values, spread)
theta_values_masked = (
    mask_invalid_values(p_values, theta_values) if theta_values is not None else None
)
entry_levels_1_masked = mask_invalid_values(p_values, entry_levels_1)
exit_levels_1_masked = mask_invalid_values(p_values, exit_levels_1)
loss_levels_1_masked = mask_invalid_values(p_values, loss_levels_1)
entry_levels_2_masked = mask_invalid_values(p_values, entry_levels_2)
exit_levels_2_masked = mask_invalid_values(p_values, exit_levels_2)
loss_levels_2_masked = mask_invalid_values(p_values, loss_levels_2)

# Count valid periods (single pvalue for both directions)
valid_count = sum(
    1
    for pv in p_values
    if pv is not None
    and not (isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD))
)
print(
    f"Valid trading periods: {valid_count}/{len(p_values)} ({100 * valid_count / len(p_values):.1f}%)"
)


# Identify no-trading regions (where pvalue is invalid)
def is_invalid(pv):
    return pv is None or (
        isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD)
    )


no_trading_regions = []
in_no_trading_region = False
region_start = None

for i in range(len(timestamps)):
    pv_invalid = is_invalid(p_values[i])

    if pv_invalid and not in_no_trading_region:
        region_start = timestamps[i]
        in_no_trading_region = True
    elif not pv_invalid and in_no_trading_region:
        no_trading_regions.append(
            (region_start, timestamps[i - 1] if i > 0 else timestamps[i])
        )
        in_no_trading_region = False
        region_start = None

# Close final region if still open
if in_no_trading_region and region_start is not None:
    no_trading_regions.append((region_start, timestamps[-1]))

print(f"Found {len(no_trading_regions)} no-trading period(s)")

# Create single figure (both directions on same plot)
fig = go.Figure()

# Add no-trading regions as shaded areas with vertical dashed borders
for idx, (start_ts, end_ts) in enumerate(no_trading_regions):
    # Add shaded region
    fig.add_vrect(
        x0=start_ts,
        x1=end_ts,
        fillcolor="gray",
        opacity=0.15,
        layer="below",
        line_width=0,
    )

    # Add vertical dashed lines at boundaries
    # Left boundary
    fig.add_vline(
        x=start_ts,
        line=dict(color="black", width=1, dash="dash"),
        opacity=0.5,
    )
    # Right boundary
    fig.add_vline(
        x=end_ts,
        line=dict(color="black", width=1, dash="dash"),
        opacity=0.5,
    )

# Plot shared spread (single line for both directions)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=spread_masked,
        mode="lines",
        name="Spread",
        line=dict(color=COLORS["spread"], width=1.5),
        opacity=0.9,
        connectgaps=False,
    )
)

# Plot mean (theta)
if theta_values_masked is not None:
    fig.add_trace(
        go.Scattergl(
            x=timestamps,
            y=theta_values_masked,
            mode="lines",
            name="Mean (θ)",
            line=dict(color=COLORS["mean"], width=1.5),
            opacity=0.7,
            connectgaps=False,
        )
    )

# Direction 1 levels (greens/blues - spread going DOWN)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=entry_levels_1_masked,
        mode="lines",
        name="Dir1 Entry (d')",
        line=dict(color=COLORS["dir1_entry"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=exit_levels_1_masked,
        mode="lines",
        name="Dir1 Exit (b')",
        line=dict(color=COLORS["dir1_exit"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=loss_levels_1_masked,
        mode="lines",
        name="Dir1 Loss (L)",
        line=dict(color=COLORS["dir1_loss"], width=1),
        connectgaps=False,
    )
)

# Direction 2 levels (reds/oranges - spread going UP)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=entry_levels_2_masked,
        mode="lines",
        name="Dir2 Entry (d')",
        line=dict(color=COLORS["dir2_entry"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=exit_levels_2_masked,
        mode="lines",
        name="Dir2 Exit (b')",
        line=dict(color=COLORS["dir2_exit"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=loss_levels_2_masked,
        mode="lines",
        name="Dir2 Loss (L)",
        line=dict(color=COLORS["dir2_loss"], width=1),
        connectgaps=False,
    )
)

# Add trades if available
if len(trades_df) > 0:
    # Direction 1 trades (greens/blues)
    dir1_trades = trades_df[trades_df["direction"] == 1]
    dir1_entries = dir1_trades[dir1_trades["type"] == "entry"]
    dir1_exits = dir1_trades[dir1_trades["type"] == "exit"]

    if len(dir1_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=dir1_entries["timestamp"].tolist(),
                y=dir1_entries["spread"].tolist(),
                mode="markers",
                name=f"Dir1 Entry ({len(dir1_entries)})",
                marker=dict(
                    color=COLORS["dir1_trade_entry"],
                    size=14,
                    symbol="triangle-down",
                    line=dict(color="darkgreen", width=2),
                ),
            )
        )

    if len(dir1_exits) > 0:
        dir1_exits_tp = dir1_exits[dir1_exits["exit_reason"] == "take_profit"]
        dir1_exits_sl = dir1_exits[dir1_exits["exit_reason"] == "stop_loss"]

        if len(dir1_exits_tp) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir1_exits_tp["timestamp"].tolist(),
                    y=dir1_exits_tp["spread"].tolist(),
                    mode="markers",
                    name=f"Dir1 TP ({len(dir1_exits_tp)})",
                    marker=dict(
                        color=COLORS["dir1_trade_tp"],
                        size=14,
                        symbol="circle",
                        line=dict(color="darkblue", width=2),
                    ),
                )
            )

        if len(dir1_exits_sl) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir1_exits_sl["timestamp"].tolist(),
                    y=dir1_exits_sl["spread"].tolist(),
                    mode="markers",
                    name=f"Dir1 SL ({len(dir1_exits_sl)})",
                    marker=dict(
                        color=COLORS["dir1_trade_sl"],
                        size=16,
                        symbol="x",
                        line=dict(width=3),
                    ),
                )
            )

        # Draw connector lines for direction 1
        if len(dir1_entries) > 0 and len(dir1_exits) > 0:
            entry_ts = dir1_entries["timestamp"].tolist()
            entry_sp = dir1_entries["spread"].tolist()
            exit_ts = dir1_exits["timestamp"].tolist()
            exit_sp = dir1_exits["spread"].tolist()
            for i in range(min(len(entry_ts), len(exit_ts))):
                fig.add_trace(
                    go.Scatter(
                        x=[entry_ts[i], exit_ts[i]],
                        y=[entry_sp[i], exit_sp[i]],
                        mode="lines",
                        line=dict(color=COLORS["dir1_connector"], width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    )
                )

    # Direction 2 trades (reds/oranges)
    dir2_trades = trades_df[trades_df["direction"] == 2]
    dir2_entries = dir2_trades[dir2_trades["type"] == "entry"]
    dir2_exits = dir2_trades[dir2_trades["type"] == "exit"]

    if len(dir2_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=dir2_entries["timestamp"].tolist(),
                y=dir2_entries["spread"].tolist(),
                mode="markers",
                name=f"Dir2 Entry ({len(dir2_entries)})",
                marker=dict(
                    color=COLORS["dir2_trade_entry"],
                    size=14,
                    symbol="triangle-up",
                    line=dict(color="darkred", width=2),
                ),
            )
        )

    if len(dir2_exits) > 0:
        dir2_exits_tp = dir2_exits[dir2_exits["exit_reason"] == "take_profit"]
        dir2_exits_sl = dir2_exits[dir2_exits["exit_reason"] == "stop_loss"]

        if len(dir2_exits_tp) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir2_exits_tp["timestamp"].tolist(),
                    y=dir2_exits_tp["spread"].tolist(),
                    mode="markers",
                    name=f"Dir2 TP ({len(dir2_exits_tp)})",
                    marker=dict(
                        color=COLORS["dir2_trade_tp"],
                        size=14,
                        symbol="circle",
                        line=dict(color="darkorange", width=2),
                    ),
                )
            )

        if len(dir2_exits_sl) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir2_exits_sl["timestamp"].tolist(),
                    y=dir2_exits_sl["spread"].tolist(),
                    mode="markers",
                    name=f"Dir2 SL ({len(dir2_exits_sl)})",
                    marker=dict(
                        color=COLORS["dir2_trade_sl"],
                        size=16,
                        symbol="x",
                        line=dict(width=3),
                    ),
                )
            )

        # Draw connector lines for direction 2
        if len(dir2_entries) > 0 and len(dir2_exits) > 0:
            entry_ts = dir2_entries["timestamp"].tolist()
            entry_sp = dir2_entries["spread"].tolist()
            exit_ts = dir2_exits["timestamp"].tolist()
            exit_sp = dir2_exits["spread"].tolist()
            for i in range(min(len(entry_ts), len(exit_ts))):
                fig.add_trace(
                    go.Scatter(
                        x=[entry_ts[i], exit_ts[i]],
                        y=[entry_sp[i], exit_sp[i]],
                        mode="lines",
                        line=dict(color=COLORS["dir2_connector"], width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    )
                )

# Update layout
fig.update_layout(
    title="Bidirectional Pairs Trading: Spread, Levels, and Trades<br><sup>Dir1 (green/blue): spread down | Dir2 (red/orange): spread up | Gray: no-trading periods</sup>",
    xaxis_title="Timestamp",
    yaxis_title="Spread Value",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.9)",
    ),
    width=1600,
    height=800,
)

# Print trade summary and take profit / stop loss statistics
if len(trades_df) > 0:
    entries_1 = trades_df[
        (trades_df["type"] == "entry") & (trades_df["direction"] == 1)
    ]
    entries_2 = trades_df[
        (trades_df["type"] == "entry") & (trades_df["direction"] == 2)
    ]

    # All exits, direction 1 and 2
    exits_1 = trades_df[(trades_df["type"] == "exit") & (trades_df["direction"] == 1)]
    exits_2 = trades_df[(trades_df["type"] == "exit") & (trades_df["direction"] == 2)]

    # For each direction, split by exit_reason
    exits_1_sl = exits_1[exits_1["exit_reason"] == "stop_loss"]
    exits_1_tp = exits_1[exits_1["exit_reason"] == "take_profit"]
    exits_2_sl = exits_2[exits_2["exit_reason"] == "stop_loss"]
    exits_2_tp = exits_2[exits_2["exit_reason"] == "take_profit"]

    print(
        f"\nDirection 1 (spread down, short close_2, long close_1): {len(entries_1)} entries, {len(exits_1_tp)} take profits, {len(exits_1_sl)} stop losses"
    )
    print(
        f"Direction 2 (spread up, short close_1, long close_2): {len(entries_2)} entries, {len(exits_2_tp)} take profits, {len(exits_2_sl)} stop losses"
    )
    print("No trades executed during this period")

fig.show()

In [ ]:
importlib.reload(stochastic)

In [ ]:
analytical = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
    mu=example["ou_mu"].to_numpy(),
    sigma=example["ou_sigma"].to_numpy(),
    theta=example["ou_theta"].to_numpy(),
    L=example["ou_theta"].to_numpy()
    - example["ou_sigma"].to_numpy() * STOP_LOSS_FACTOR,
    r=0.01,
    c=0.001,
    use_analytical=True,
)
analytical

In [ ]:
numerical = stochastic.OrnsteinUhlenbeck.F(
    (example["ou_theta"] - example["ou_sigma"] * STOP_LOSS_FACTOR).to_numpy(),
    example["ou_mu"].to_numpy(),
    example["ou_sigma"].to_numpy(),
    example["ou_theta"].to_numpy(),
    0.01,
    use_analytical=False,
)
numerical

In [ ]:
"""
Code to plot how numerical vs analytical F(x) relationship changes across different inputs.
Can be added as a notebook cell.
"""

import sys
from pathlib import Path

# Add the workspace root to Python path
workspace_root = Path.cwd().parent if Path.cwd().name != "mc-notebooks" else Path.cwd()
sys.path.insert(0, str(workspace_root))

import src.utils.stochastic as stochastic

# Parameters
r = 0.0001
sigma = 0.00023
theta = 0.000245

# Test different mu values to see how relationship changes with alpha
mu_values = np.array([0.00001, 0.00005, 0.0001, 0.001, 0.01, 0.05, 0.1])
x_values = np.linspace(theta - 0.002, theta + 0.002, 100)

# Collect all data
all_analytical = []
all_numerical = []
alpha_by_point = []

for mu in mu_values:
    alpha = (r / mu) - 1
    for x in x_values:
        try:
            numerical = stochastic.OrnsteinUhlenbeck.F(
                x, mu, sigma, theta, r, use_analytical=False
            )
            analytical = stochastic.OrnsteinUhlenbeck.F(
                x, mu, sigma, theta, r, use_analytical=True
            )

            # Convert to float if needed
            if isinstance(numerical, np.ndarray):
                numerical = float(numerical.item())
            if isinstance(analytical, np.ndarray):
                analytical = float(analytical.item())

            all_numerical.append(numerical)
            all_analytical.append(analytical)
            alpha_by_point.append(alpha)
        except Exception:
            pass

all_numerical = np.array(all_numerical)
all_analytical = np.array(all_analytical)
alpha_by_point = np.array(alpha_by_point)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "How Numerical vs Analytical Relationship Changes Across Inputs", fontsize=14
)

# Plot 1: Main scatter plot (like your example)
ax1 = axes[0, 0]
ax1.plot(
    all_numerical,
    all_analytical,
    "o",
    alpha=0.5,
    markersize=3,
    label="analytical vs numerical",
)
ax1.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax1.set_xlabel("Numerical F(x)")
ax1.set_ylabel("Analytical F(x)")
ax1.set_title("Analytical vs Numerical F(x)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Color-coded by alpha regime
ax2 = axes[0, 1]
scatter = ax2.scatter(
    all_numerical, all_analytical, c=alpha_by_point, cmap="viridis", alpha=0.6, s=20
)
ax2.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax2.set_xlabel("Numerical F(x)")
ax2.set_ylabel("Analytical F(x)")
ax2.set_title("Colored by α = r/μ - 1")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax2, label="α")

# Plot 3: Show relationship for different alpha regimes separately
ax3 = axes[1, 0]
colors = plt.cm.viridis(np.linspace(0, 1, len(mu_values)))
for i, mu in enumerate(mu_values):
    alpha = (r / mu) - 1
    mask = alpha_by_point == alpha
    if np.any(mask):
        ax3.plot(
            all_numerical[mask],
            all_analytical[mask],
            "o-",
            color=colors[i],
            alpha=0.6,
            markersize=2,
            label=f"α={alpha:.2f}",
        )

ax3.plot(
    [all_numerical.min(), all_numerical.max()],
    [all_numerical.min(), all_numerical.max()],
    "r--",
    linewidth=2,
    label="Perfect match",
)
ax3.set_xlabel("Numerical F(x)")
ax3.set_ylabel("Analytical F(x)")
ax3.set_title("By Alpha Regime")
ax3.legend(fontsize=8, ncol=2, loc="upper left")
ax3.grid(True, alpha=0.3)

# Plot 4: Error analysis
ax4 = axes[1, 1]
errors = np.abs(all_analytical - all_numerical) / np.abs(all_numerical) * 100
ax4.scatter(all_numerical, errors, c=alpha_by_point, cmap="plasma", alpha=0.6, s=20)
ax4.set_xlabel("Numerical F(x)")
ax4.set_ylabel("Relative Error (%)")
ax4.set_title("Error vs F(x) Value (colored by α)")
ax4.set_yscale("log")
ax4.grid(True, alpha=0.3)
plt.colorbar(ax4.collections[0], ax=ax4, label="α")

plt.tight_layout()
plt.show()

# Print summary
print("\nSummary:")
print(f"  Total points: {len(all_numerical)}")
print(f"  Mean error: {np.mean(errors):.2f}%")
print(f"  Median error: {np.median(errors):.2f}%")
print(f"  Max error: {np.max(errors):.2f}%")
print(
    f"  Points with <1% error: {np.sum(errors < 1)} ({100 * np.sum(errors < 1) / len(errors):.1f}%)"
)
print(
    f"  Points with <5% error: {np.sum(errors < 5)} ({100 * np.sum(errors < 5) / len(errors):.1f}%)"
)

In [ ]:
plt.plot(numerical, label="numerical")
plt.plot(analytical, label="analytical")
plt.legend()
plt.show()

In [ ]:
pairs_trading_fitted_frame_computed["spread"] = (
    pairs_trading_fitted_frame_computed["close_1"]
    - pairs_trading_fitted_frame_computed["alpha"]
    - pairs_trading_fitted_frame_computed["beta"]
    * pairs_trading_fitted_frame_computed["close_2"]
)
pairs_trading_fitted_frame_computed

In [ ]:
# Compute entry and exit levels in parallel on resampled data
@delayed
def compute_entry_exit_levels_chunk(resampled_chunk: pd.DataFrame) -> pd.DataFrame:
    """
    Compute entry and exit levels for a chunk of resampled data.

    Args:
        resampled_chunk: DataFrame with columns: timestamp, provider_asset_group_id,
                        ou_mu, ou_theta, ou_sigma, and other OU parameters

    Returns:
        DataFrame with added columns: ou_L, entry_level, exit_level
    """
    result = resampled_chunk.copy()

    # Compute ou_L (loss level)
    result["ou_L"] = result["ou_theta"] - result["ou_sigma"] * STOP_LOSS_FACTOR

    # Initialize entry and exit levels
    result["entry_level"] = np.nan
    result["exit_level"] = np.nan

    # Compute entry and exit levels for rows with valid OU parameters
    valid_mask = (
        result["ou_mu"].notna()
        & result["ou_sigma"].notna()
        & result["ou_theta"].notna()
        & (result["ou_sigma"] > 0)
    )

    if valid_mask.any():
        valid_rows = result[valid_mask]

        for idx, row in valid_rows.iterrows():
            try:
                ou_mu = row["ou_mu"]
                ou_sigma = row["ou_sigma"]
                ou_theta = row["ou_theta"]
                ou_L = row["ou_L"]

                # Compute exit level first (needed for entry level)
                exit_level = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
                    mu=ou_mu,
                    sigma=ou_sigma,
                    theta=ou_theta,
                    r=DISCOUNT_RATE,
                    c=TRANSACTION_COST,
                    L=ou_L,
                )

                # Compute entry level (uses exit level internally)
                entry_level = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
                    mu=ou_mu,
                    sigma=ou_sigma,
                    theta=ou_theta,
                    r=DISCOUNT_RATE,
                    c=TRANSACTION_COST,
                    L=ou_L,
                    exit_level=exit_level,  # Pass pre-computed exit level for efficiency
                )

                result.at[idx, "entry_level"] = entry_level
                result.at[idx, "exit_level"] = exit_level
            except Exception:
                # If computation fails, leave as NaN
                pass

    return result


# Create resampled data (similar to cell 16)
pairs_trading_resampled = (
    pairs_trading_fitted_frame_computed.drop(columns=["close_1", "close_2"])
    .set_index("timestamp")
    .groupby("provider_asset_group_id")
    .resample("1D")
    .first()
    .sort_values("timestamp")
    .drop(columns=["provider_asset_group_id"])
    .reset_index()
)

# Split into chunks for parallel processing
provider_asset_group_ids = pairs_trading_resampled["provider_asset_group_id"].unique()
n_chunks = min(N_WORKERS, len(provider_asset_group_ids))
group_chunks = np.array_split(provider_asset_group_ids, n_chunks)

# Create delayed tasks
delayed_chunks = []
for chunk in group_chunks:
    chunk_data = pairs_trading_resampled[
        pairs_trading_resampled["provider_asset_group_id"].isin(chunk)
    ].copy()
    delayed_chunks.append(compute_entry_exit_levels_chunk(chunk_data))

# Compute in parallel
print("Computing entry and exit levels in parallel...")
pairs_trading_resampled_with_levels = pd.concat(
    [chunk.compute() for chunk in delayed_chunks], ignore_index=True
)
print(f"Computed levels for {len(pairs_trading_resampled_with_levels)} rows")
pairs_trading_resampled_with_levels

In [ ]:
pairs_trading_frame_computed = pairs_trading_frame_computed.reset_index(
    level="provider_asset_group_id"
)
pairs_trading_frame_computed

In [ ]:
pairs_trading_resampled_frame["spread"] = (
    pairs_trading_resampled_frame["close_1"]
    - pairs_trading_resampled_frame["alpha"]
    - pairs_trading_resampled_frame["beta"] * pairs_trading_resampled_frame["close_2"]
)
pairs_trading_resampled_frame = pairs_trading_resampled_frame.loc[
    pairs_trading_resampled_frame["alpha"].notna()
]
pairs_trading_resampled_frame

In [ ]:
pairs_trading_resampled_frame.loc[
    (pairs_trading_resampled_frame["p_value"] < 0.01)
    & (pairs_trading_resampled_frame["ou_sigma"] > 0.01)
]

In [ ]:
backtest_frame = pairs_trading_frame_computed.reset_index(
    level="provider_asset_group_id"
)

In [ ]:
backtest_frame = pd.merge_asof(
    backtest_frame[["timestamp", "provider_asset_group_id", "close_1", "close_2"]]
    .drop_duplicates()
    .sort_values("timestamp"),
    backtest_frame.drop(columns=["close_1", "close_2"]).sort_values("timestamp"),
    on="timestamp",
    by="provider_asset_group_id",
)
backtest_frame = backtest_frame.loc[backtest_frame["alpha"].notna()]
backtest_frame["spread"] = (
    backtest_frame["close_1"]
    - backtest_frame["alpha"]
    - backtest_frame["beta"] * backtest_frame["close_2"]
)
backtest_frame

In [ ]:
import plotly.graph_objects as go

example = 57
example_df = backtest_frame.loc[backtest_frame["provider_asset_group_id"] == example]

# Ensure timestamp is datetime for finer x-axis control
example_df["timestamp"] = pd.to_datetime(example_df["timestamp"])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=example_df["timestamp"], y=example_df["spread"], name="spread", mode="lines"
    )
)
fig.add_trace(
    go.Scatter(
        x=example_df["timestamp"],
        y=example_df["theta"],
        name="theta",
        mode="lines",
    )
)

# Add entry, exit, and loss levels if they exist
if "entry_level" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["entry_level"],
            name="entry_level",
            mode="lines",
            line=dict(color="green", width=1.5, dash="dash"),
        )
    )

if "exit_level" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["exit_level"],
            name="exit_level",
            mode="lines",
            line=dict(color="blue", width=1.5, dash="dash"),
        )
    )

if "ou_L" in example_df.columns:
    fig.add_trace(
        go.Scatter(
            x=example_df["timestamp"],
            y=example_df["loss_level"],
            name="loss_level (ou_L)",
            mode="lines",
            line=dict(color="red", width=1.5, dash="dash"),
        )
    )

import numpy as np

n = len(example_df["timestamp"])
if n >= 8:
    tick_indices = np.linspace(0, n - 1, 8, dtype=int)
    tickvals = example_df["timestamp"].iloc[tick_indices]
else:
    tickvals = example_df["timestamp"]

# Format tick labels as nice datetimes
ticktext = [t.strftime("%Y-%m-%d %H:%M") for t in tickvals]

fig.update_layout(
    title=f"Spread, OU Theta, and Trading Levels for provider_asset_group_id {example}",
    xaxis_title="Timestamp",
    yaxis_title="Value",
    xaxis=dict(
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=45,  # Nicely angled for readability
    ),
)
fig.show()

In [ ]:
example_df["L"] = (
    example_df["residual_mean"] - example_df["residual_std"] * STOP_LOSS_FACTOR
)
example_dd

In [ ]:
cointegrated_provider_asset_group_ids = pairs_trading_fitted_frame_computed.loc[
    pairs_trading_fitted_frame_computed["p_value"] < 0.001
].index.tolist()
print(
    f"Cointegrated provider asset group ids (count: {len(cointegrated_provider_asset_group_ids)}): {cointegrated_provider_asset_group_ids}"
)

In [ ]:
def get_cointegrated_stats(df: pd.DataFrame) -> pd.Series:
    """
    Get the cointegrated stats for a given dataframe.
    """

    # Compute the linear regression.
    X = df["close_1"].to_numpy()
    y = df["close_2"].to_numpy()
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()

    # Get the residuals.
    linear_fit_alpha = results.params[0]
    linear_fit_beta = results.params[1]
    linear_fit_mse = results.mse_total
    linear_fit_r_squared = results.rsquared
    linear_fit_r_squared_adj = results.rsquared_adj
    residuals = results.resid

    # Get the cointegration stats.
    ou_params = OrnsteinUhlenbeck().fit(residuals)

    return pd.Series(
        [
            linear_fit_alpha,
            linear_fit_beta,
            linear_fit_mse,
            linear_fit_r_squared,
            linear_fit_r_squared_adj,
            ou_params.mu,
            ou_params.theta,
            ou_params.sigma,
        ],
        index=[
            "linear_fit_alpha",
            "linear_fit_beta",
            "linear_fit_mse",
            "linear_fit_r_squared",
            "linear_fit_r_squared_adj",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
        ],
        dtype=float,
    )

In [ ]:
# Broadcast market data to all workers
print("Broadcasting market data to workers...")
market_data_future = client.scatter(market_data, broadcast=True)
print("Market data broadcasted successfully")

In [ ]:
cointegrated_pairs_trading_frame = get_pairs_trading_frame(
    start_naive,
    end_naive,
    cointegrated_provider_asset_group_ids,
    members_data,
    market_data_future,
    N_WORKERS,
)

In [ ]:
cointegrated_pairs_trading_stats = cointegrated_pairs_trading_frame.groupby(
    "provider_asset_group_id"
)[["close_1", "close_2"]].apply(
    lambda df: get_cointegrated_stats(df),
    meta={
        "linear_fit_alpha": pd.Series([], dtype=float),
        "linear_fit_beta": pd.Series([], dtype=float),
        "linear_fit_mse": pd.Series([], dtype=float),
        "linear_fit_r_squared": pd.Series([], dtype=float),
        "linear_fit_r_squared_adj": pd.Series([], dtype=float),
        "ou_mu": pd.Series([], dtype=float),
        "ou_theta": pd.Series([], dtype=float),
        "ou_sigma": pd.Series([], dtype=float),
    },
)

In [ ]:
cointegrated_pairs_trading_stats_computed = cointegrated_pairs_trading_stats.compute()
cointegrated_pairs_trading_stats_computed

In [ ]:
# cointegration_p_values_computed.to_csv("cointegration_p_values.csv")
# cointegrated_pairs_trading_stats_computed.to_csv("cointegrated_pairs_trading_stats.csv")
# cointegration_p_values_computed.to_parquet("cointegration_p_values.parquet")
# cointegrated_pairs_trading_stats_computed.to_parquet(
#     "cointegrated_pairs_trading_stats.parquet"
# )


In [ ]:
toset = cointegration_p_values_computed.merge(
    cointegrated_pairs_trading_stats_computed, left_index=True, right_index=True
).reset_index()
toset = toset.rename(columns={"p_value": "cointegration_p_value"})
toset["lookback_window_seconds"] = 30 * 24 * 60 * 60
toset["timestamp"] = end_naive
toset = toset[
    [
        "timestamp",
        "provider_asset_group_id",
        "lookback_window_seconds",
        "cointegration_p_value",
        "linear_fit_alpha",
        "linear_fit_beta",
        "linear_fit_mse",
        "linear_fit_r_squared",
        "linear_fit_r_squared_adj",
        "ou_mu",
        "ou_theta",
        "ou_sigma",
    ]
]
toset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

data = (
    cointegrated_pairs_trading_frame.reset_index()
    .compute()
    .merge(
        toset.drop(columns=["lookback_window_seconds", "timestamp"]),
        on="provider_asset_group_id",
    )
)
data = data.loc[
    data["provider_asset_group_id"] == cointegrated_provider_asset_group_ids[0]
]
timestamps = data["timestamp"].to_numpy()
close_1 = data["close_1"].to_numpy()
close_2 = data["close_2"].to_numpy()
residuals = close_2 - data["linear_fit_alpha"] - data["linear_fit_beta"] * close_1

# Get the provider asset group members with asset symbols

from_asset = aliased(models.Asset)
to_asset = aliased(models.Asset)

pair_df = pd.read_sql(
    select(
        models.ProviderAssetGroupMember.order,
        from_asset.symbol.label("from_asset_symbol"),
        to_asset.symbol.label("to_asset_symbol"),
    )
    .join(from_asset, models.ProviderAssetGroupMember.from_asset_id == from_asset.id)
    .join(to_asset, models.ProviderAssetGroupMember.to_asset_id == to_asset.id)
    .where(
        models.ProviderAssetGroupMember.provider_asset_group_id
        == cointegrated_provider_asset_group_ids[0]
    ),
    engine,
)

# Get the symbols for the title
from_asset_1 = pair_df[pair_df["order"] == 1]["from_asset_symbol"].iloc[0]
from_asset_2 = pair_df[pair_df["order"] == 2]["from_asset_symbol"].iloc[0]
to_asset_1 = pair_df[pair_df["order"] == 1]["to_asset_symbol"].iloc[0]
to_asset_2 = pair_df[pair_df["order"] == 2]["to_asset_symbol"].iloc[0]

# Create figure and plot with nice date formatting
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(timestamps, residuals)
ax.set_title(
    f"Residuals for {to_asset_1}-{from_asset_1}/{to_asset_2}-{from_asset_2} (Pair ID: {cointegrated_provider_asset_group_ids[0]})"
)
ax.set_xlabel("Date")
ax.set_ylabel("Residuals")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.show()

In [ ]:
# set_data(
#     engine,
#     models.ProviderAssetGroupAttribute.__tablename__,
#     toset,
#     operation_type="upsert",
# )

In [ ]:
# cluster.close(force_shutdown=True)